In [1]:
pip install pathlib tqdm datasets pandas soundfile librosa seaborn jiwer evaluate

  Using cached pathlib-1.0.1-py3-none-any.whl.metadata (5.1 kB)
  Using cached datasets-5.0.0-py3-none-any.whl.metadata (23 kB)
  Using cached soundfile-0.14.0-py2.py3-none-manylinux_2_28_x86_64.whl.metadata (18 kB)
  Using cached librosa-0.11.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached jiwer-4.0.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached evaluate-0.4.6-py3-none-any.whl.metadata (9.5 kB)
  Using cached pyarrow-24.0.0-cp310-cp310-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached xxhash-3.8.0-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (14 kB)
  Using cached multiprocess-0.70.19-py310-none-any.whl.metadata (7.5 kB)
  Using cached aiohttp-3.14.1-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (8.3 kB)
  Using cached audioread-3.1.0-py3-none-any.whl.metad

In [2]:
!pip uninstall -y datasets torchcodec
!pip install "datasets==3.6.0"

Found existing installation: datasets 5.0.0
Uninstalling datasets-5.0.0:
  Successfully uninstalled datasets-5.0.0
  Using cached datasets-3.6.0-py3-none-any.whl.metadata (19 kB)
  Using cached dill-0.3.8-py3-none-any.whl.metadata (10 kB)
  Using cached multiprocess-0.70.16-py310-none-any.whl.metadata (7.2 kB)
  Using cached fsspec-2025.3.0-py3-none-any.whl.metadata (11 kB)
Using cached datasets-3.6.0-py3-none-any.whl (491 kB)
Using cached dill-0.3.8-py3-none-any.whl (116 kB)
Using cached fsspec-2025.3.0-py3-none-any.whl (193 kB)
Using cached multiprocess-0.70.16-py310-none-any.whl (134 kB)
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2026.4.0
    Uninstalling fsspec-2026.4.0:
      Successfully uninstalled fsspec-2026.4.0
  Attempting uninstall: dill
    Found existing installation: dill 0.4.1
    Uninstalling dill-0.4.1:
      Successfully uninstalled dill-0.4.1
  Attempting uninstall: multiprocessm━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [dill]
    Found existing 

In [3]:
import torch
import re
import jiwer
from transformers.models.whisper.english_normalizer import BasicTextNormalizer
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from datasets import load_from_disk
from evaluate import load

In [9]:
device = "cuda:0"

model_id = "openai/whisper-small"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    low_cpu_mem_usage=True,
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    chunk_length_s=30,
    batch_size=128,  # batch size for inference - set based on your device
    device=device,
)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


In [10]:
dss = {
    "myst": load_from_disk("/home/myst-v0.4.2/myst_dataset.ds"),
}

Loading dataset from disk:   0%|          | 0/23 [00:00<?, ?it/s]

In [11]:
sample = dss["myst"]["test"][0]["audio"]

result = pipe(sample)

print(dss["myst"]["test"][0]["transcription"])
print(result)

GOOD HOW ARE YOU
{'text': ' Good. How are you?'}


# Run on all Data

In [12]:
results = pipe(dss["myst"]["test"]["audio"])

OutOfMemoryError: CUDA out of memory. Tried to allocate 564.00 MiB. GPU 0 has a total capacity of 139.80 GiB of which 391.12 MiB is free. Including non-PyTorch memory, this process has 139.41 GiB memory in use. Of the allocated memory 138.57 GiB is allocated by PyTorch, and 174.84 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [28]:
from text2digits import text2digits
import string

t2d = text2digits.Text2Digits()
punctuation_remover = str.maketrans("", "", string.punctuation)


def normalize_transcript(text):
    # The original transcript has annotations, for example a pause is <pau>
    # Remove tags in angle brackets
    text = re.sub(r"<[^>]*>", "", text)

    # These are "false starts" in the original transcript, for example th*
    # These are ignored by ASR
    # Remove words that end with asterisks (e.g., th*)
    text = re.sub(r"\S*\*", "", text)

    # Remove all punctuation
    text = text.translate(punctuation_remover)

    # Clean up excess spaces in the original transcript or resulting from above operations
    text = re.sub(r"\s+", " ", text)

    # Convert number representations, e.g., "thirteen" to "13"
    # This is imperfect (does not know when "one" is a pronoun vs. a number)
    # But we apply the same normalization to both samples, so works fine for Word Error Rate
    try:
        normalized_text = t2d.convert(text)
    except:
        print(text)
    return normalized_text.strip().lower()

In [30]:
wer = load("wer")
predictions = [normalize_transcript(d["text"]) for d in results]
references = [normalize_transcript(text) for text in dss["myst"]["sentence"]]
wer_score = wer.compute(predictions=predictions, references=references)
print(wer_score)  # 0.20958586984480837 on 07-08-25

0.20958586984480837


In [6]:
normalizer = BasicTextNormalizer()


def normalize_transcript(text):
    # The original transcript has annotations, for example a pause is <pau>
    # Remove tags in angle brackets
    text = re.sub(r"<[^>]*>", "", text)

    # These are "false starts" in the original transcript, for example th*
    # These are ignored by ASR
    # Remove words that end with asterisks (e.g., th*)
    text = re.sub(r"\S*\*", "", text)

    # Apply Whisper's English normalizer
    normalized_text = normalizer(text)

    return normalized_text

In [7]:
def weighted_wer(ref: list[str], pred: list[str]):
    # Normalize both predictions and references
    pred_normalized = [normalize_transcript(text) for text in pred]
    label_normalized = [normalize_transcript(text) for text in ref]

    total_errors = 0
    total_words = 0

    for pred_text, ref_text in zip(pred_normalized, label_normalized):
        ref_words = ref_text.split()

        # Compute WER for this sample
        sample_wer = jiwer.wer(ref_text, pred_text)

        # Accumulate weighted errors
        sample_errors = sample_wer * len(ref_words)
        total_errors += sample_errors
        total_words += len(ref_words)

    weighted_wer = total_errors / total_words if total_words > 0 else 0.0

    return {"wer": weighted_wer}


predictions = [d["text"] for d in results]
references = dss["myst"]["sentence"]
wer_score = weighted_wer(references, predictions)
print(wer_score)

{'wer': 0.2082938074963048}
